<a href="https://colab.research.google.com/github/jarekwan/PROJEKT_SCANNER/blob/main/modul_pobieranie.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
os.makedirs('/content/drive/MyDrive/projekt_test', exist_ok=True)

print("folder ready")

Mounted at /content/drive
folder ready


In [ ]:
%%writefile /content/drive/MyDrive/projekt_test/modul_pobieranie.py
# -*- coding: utf-8 -*-

import json
from pathlib import Path

import pandas as pd
import requests


def run():
    zrd = input("alpha czy twelve: ").strip().lower()
    tik = input("ticker: ").strip().upper()

    fd = Path("/content/drive/MyDrive/projekt_test")

    if zrd == "alpha":
        kl = "GKIB5CELANXY4KFA"
        url = "https://www.alphavantage.co/query"
        par = {
            "function": "TIME_SERIES_DAILY",
            "symbol": tik,
            "outputsize": "full",
            "apikey": kl
        }

        r = requests.get(url, params=par, timeout=30)
        r.raise_for_status()
        d = r.json()

        fp = fd / f"{tik}_alpha.json"
        with open(fp, "w", encoding="utf-8") as f:
            json.dump(d, f, ensure_ascii=False, indent=2)

        print("glowne klucze odpowiedzi:", list(d.keys()))

        if "Error Message" in d:
            print("ticker nie istnieje albo alpha go nie rozpoznaje")
            print(d["Error Message"])
            return

        if "Note" in d:
            print("limit albo uwaga z alpha vantage")
            print(d["Note"])
            return

        if "Information" in d:
            print("informacja z alpha vantage:")
            print(d["Information"])
            return

        if "Meta Data" not in d or "Time Series (Daily)" not in d:
            print("brak oczekiwanych sekcji w odpowiedzi alpha")
            return

        sr = d["Time Series (Daily)"]

        if not sr:
            print("ticker istnieje, ale brak danych dziennych")
            return

        pierwsza_data = next(iter(sr))
        pierwszy_rekord = sr[pierwsza_data]

        print("przykladowa data:", pierwsza_data)
        print("klucze w rekordzie:", list(pierwszy_rekord.keys()))

        df = pd.DataFrame.from_dict(sr, orient="index")
        df.index.name = "data"
        df = df.reset_index()

        print("kolumny przed zmiana nazw:", list(df.columns))

        mapa = {
            "1. open": "open",
            "2. high": "high",
            "3. low": "low",
            "4. close": "close",
            "5. volume": "volume"
        }

        df = df.rename(columns=mapa)

        ocz = ["data", "open", "high", "low", "close", "volume"]
        brak = [k for k in ocz if k not in df.columns]

        if brak:
            print("brakuje kolumn:", brak)
            print("rzeczywiste kolumny po zmianie nazw:", list(df.columns))
            return

        df = df[ocz]
        print("zapisano:", fp)
        display(df.head(10))

    elif zrd == "twelve":
        kl = "0ae01ea2dc5a43b58a1514eebfdf7ee8"
        url = "https://api.twelvedata.com/time_series"
        par = {
            "symbol": tik,
            "interval": "1day",
            "outputsize": 5000,
            "apikey": kl
        }

        r = requests.get(url, params=par, timeout=30)
        r.raise_for_status()
        d = r.json()

        fp = fd / f"{tik}_twelve.json"
        with open(fp, "w", encoding="utf-8") as f:
            json.dump(d, f, ensure_ascii=False, indent=2)

        print("glowne klucze odpowiedzi:", list(d.keys()))

        if d.get("status") == "error":
            print("ticker nie istnieje albo twelve go nie rozpoznaje")
            print(d.get("message", "nieznany blad"))
            return

        if "values" not in d:
            print("brak sekcji values w odpowiedzi twelve")
            return

        ls = d["values"]

        if not ls:
            print("ticker istnieje, ale brak danych dziennych")
            return

        print("klucze w rekordzie:", list(ls[0].keys()))

        df = pd.DataFrame(ls)

        print("kolumny przed zmiana nazw:", list(df.columns))

        df = df.rename(columns={"datetime": "data"})

        ocz = ["data", "open", "high", "low", "close", "volume"]
        brak = [k for k in ocz if k not in df.columns]

        if brak:
            print("brakuje kolumn:", brak)
            print("rzeczywiste kolumny po zmianie nazw:", list(df.columns))
            return

        df = df[ocz]
        print("zapisano:", fp)
        display(df.head(10))

    else:
        print("zle zrodlo")

Overwriting /content/drive/MyDrive/projekt_test/modul_pobieranie.py
